# SafeSpan Bridge Analytics — EDA & Data Cleaning### Predictive Maintenance for U.S. Bridge Infrastructure**Dataset**: National Bridge Inventory (NBI) — 5,000,000+ records (2018–2025)  **Target**:  → Critical, Poor, Fair, Good---

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')sns.set_theme(style='whitegrid', palette='viridis')plt.rcParams['figure.figsize'] = (14, 6)plt.rcParams['font.size'] = 12print('All libraries loaded successfully.')

## 1. Load the DatasetWe load in chunks to handle the 2GB+ file efficiently.

In [ ]:
# Load the dataset — use a sample for EDA speed, full data for final training
# NOTE: Since the file is sorted by year, nrows=500_000 only reads the first year.
# To see trends, you need to read more rows or use a random sample.
SAMPLE_SIZE = 2_000_000  # Increase this to 2M or 3M to see multiple years

df = pd.read_csv("nbi_5million.csv", nrows=SAMPLE_SIZE, low_memory=False)
print(f"Loaded {SAMPLE_SIZE:,} rows. Unique years: {sorted(df["YEAR"].unique().tolist())}")
print(f"Shape: {df.shape}")

## 2. Initial Data Overview

In [ ]:
# Basic infoprint('=== Column Types ===')print(df.dtypes.value_counts())print(f'\nTotal Columns: {df.shape[1]}')print(f'Total Rows: {df.shape[0]:,}')

In [ ]:
# Target distributionprint('=== Target Variable Distribution ===')target_counts = df['TARGET_CONDITION'].value_counts()print(target_counts)print(f'\nPercentages:')print((target_counts / len(df) * 100).round(2))

### 2a. Target Class DistributionThis is the most critical plot — it reveals the severe class imbalance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Bar chartcolors = {'Critical': '#e74c3c', 'Poor': '#e67e22', 'Fair': '#f39c12', 'Good': '#27ae60'}order = ['Critical', 'Poor', 'Fair', 'Good']target_counts = df['TARGET_CONDITION'].value_counts().reindex(order)axes[0].bar(target_counts.index, target_counts.values, color=[colors[c] for c in order], edgecolor='black')axes[0].set_title('Bridge Condition Class Distribution', fontsize=14, fontweight='bold')axes[0].set_ylabel('Count')for i, v in enumerate(target_counts.values):    axes[0].text(i, v + len(df)*0.005, f'{v:,}', ha='center', fontweight='bold')# Pie chartaxes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',            colors=[colors[c] for c in order], startangle=90, explode=[0.1, 0.05, 0, 0])axes[1].set_title('Condition Distribution (%)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('plots/01_target_distribution.png', dpi=150, bbox_inches='tight')plt.show()print('INSIGHT: Severe class imbalance — Critical class is extremely rare but most important to predict.')

## 3. Missing Value Analysis

In [ ]:
# Calculate missing percentagesmissing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)missing_df = pd.DataFrame({'Column': missing.index, 'Missing %': missing.values})missing_df = missing_df[missing_df['Missing %'] > 0]print(f'Columns with missing values: {len(missing_df)} out of {df.shape[1]}')print(f'\nTop 20 columns by missing %:')print(missing_df.head(20).to_string(index=False))

In [ ]:
# Missing value heatmap (top 30 columns with most missing)top_missing = missing[missing > 0].head(30)if len(top_missing) > 0:    fig, ax = plt.subplots(figsize=(14, 8))    ax.barh(top_missing.index[::-1], top_missing.values[::-1], color='coral', edgecolor='black')    ax.set_xlabel('Missing %')    ax.set_title('Top 30 Columns by Missing Value Percentage', fontsize=14, fontweight='bold')    ax.axvline(x=90, color='red', linestyle='--', label='90% threshold (drop)')    ax.legend()    plt.tight_layout()    plt.savefig('plots/02_missing_values.png', dpi=150, bbox_inches='tight')    plt.show()

## 4. Data Cleaning### 4a. Drop Useless ColumnsColumns that are 90%+ null, free-text identifiers, or date strings.

In [ ]:
# Columns to drop — high missing or identifiersdrop_cols_missing = [col for col in df.columns if df[col].isnull().mean() > 0.90]drop_cols_identifiers = [    'STRUCTURE_NUMBER_008',    # Unique bridge ID    'FEATURES_DESC_006A',      # Free text    'FACILITY_CARRIED_007',    # Free text    'LOCATION_009',            # Free text    'DATE_LAST_UPDATE',        # Date string    'DATE_OF_INSPECT_090',     # Date string    'FED_AGENCY',              # Sparse identifier    'LRS_INV_ROUTE_013A',      # Sparse identifier    'DEDUCT_CODE',             # Administrative    'TYPE_LAST_UPDATE',        # Administrative]# Combine and deduplicatecols_to_drop = list(set(drop_cols_missing + drop_cols_identifiers))cols_to_drop = [c for c in cols_to_drop if c in df.columns]print(f'Dropping {len(cols_to_drop)} useless/sparse columns:')for c in sorted(cols_to_drop):    pct = df[c].isnull().mean() * 100    print(f'  {c:50s} (missing: {pct:.1f}%)')df.drop(columns=cols_to_drop, inplace=True)print(f'\nShape after dropping useless columns: {df.shape}')

### 4b. ⚠️ CRITICAL — Remove Data Leakage ColumnsThese columns are derived from or directly encode the condition rating. Using them would be cheating.

In [ ]:
leakage_cols = [    'DECK_COND_058',             # This IS the raw target (0-9 rating)    'SUPERSTRUCTURE_COND_059',   # This IS the raw target    'SUBSTRUCTURE_COND_060',     # This IS the raw target    'CHANNEL_COND_061',          # Used to compute target    'CULVERT_COND_062',          # Used to compute target    'STRUCTURAL_EVAL_067',       # Derived from condition ratings    'DECK_GEOMETRY_EVAL_068',    # Evaluation score (potential leakage)    'UNDCLRENCE_EVAL_069',       # Evaluation score    'POSTING_EVAL_070',          # Derived from structural eval    'WATERWAY_EVAL_071',         # Evaluation score    'APPR_ROAD_EVAL_072',        # Evaluation score    'SUFFICIENCY_RATING',        # Composite score using condition ratings    'STATUS_WITH_10YR_RULE',     # Based on sufficiency rating    'STATUS_NO_10YR_RULE',       # Based on sufficiency rating    'CAT10', 'CAT23', 'CAT29',  # Derived categorizations]leakage_cols = [c for c in leakage_cols if c in df.columns]print(f'Removing {len(leakage_cols)} leakage-prone columns:')for c in leakage_cols:    print(f'  ❌ {c}')df.drop(columns=leakage_cols, inplace=True)print(f'\nShape after removing leakage: {df.shape}')

### 4c. Handle Mixed TypesSeveral columns contain "N" (Not Applicable), "U" (Unknown), or "T" mixed with numeric values.

In [ ]:
# Fix SCOUR_CRITICAL_113 — contains N, U, T mixed with 0-9if 'SCOUR_CRITICAL_113' in df.columns:    print('SCOUR_CRITICAL_113 unique values BEFORE cleaning:')    print(df['SCOUR_CRITICAL_113'].value_counts())        df['SCOUR_CRITICAL_113'] = pd.to_numeric(        df['SCOUR_CRITICAL_113'].replace({'N': np.nan, 'U': np.nan, 'T': np.nan}),        errors='coerce'    )print(f'\nAfter cleaning: {df["SCOUR_CRITICAL_113"].notna().sum():,} valid numeric values')# Fix other potentially mixed columnsmixed_cols = ['OPEN_CLOSED_POSTED_041', 'NAVIGATION_038']for col in mixed_cols:    if col in df.columns and df[col].dtype == 'object':        original_nunique = df[col].nunique()        df[col] = pd.to_numeric(df[col], errors='coerce')        print(f'{col}: converted to numeric (was {original_nunique} unique values)')

### 4d. Drop Rows with Unknown Target

In [ ]:
before = len(df)df = df[df['TARGET_CONDITION'] != 'Unknown'].copy()after = len(df)print(f'Dropped {before - after:,} rows with Unknown target')print(f'Remaining: {after:,} rows')print(f'\nFinal target distribution:')print(df['TARGET_CONDITION'].value_counts())

### 4e. Missing Value ImputationStrategy: Median for numeric, Mode for categorical, plus missing indicator flags.

In [ ]:
# Separate numeric and categorical columns (excluding target)feature_cols = [c for c in df.columns if c not in ['TARGET_CONDITION', 'YEAR']]numeric_cols = df[feature_cols].select_dtypes(include='number').columns.tolist()categorical_cols = df[feature_cols].select_dtypes(include='object').columns.tolist()print(f'Numeric features: {len(numeric_cols)}')print(f'Categorical features: {len(categorical_cols)}')# Add missing indicator flags for columns with >5% missingfor col in numeric_cols + categorical_cols:    pct_missing = df[col].isnull().mean()    if pct_missing > 0.05:        df[f'{col}_MISSING'] = df[col].isnull().astype(int)        print(f'  Created missing indicator: {col}_MISSING ({pct_missing*100:.1f}% missing)')# Impute numeric with medianfor col in numeric_cols:    if df[col].isnull().any():        median_val = df[col].median()        df[col].fillna(median_val, inplace=True)# Impute categorical with modefor col in categorical_cols:    if df[col].isnull().any():        mode_val = df[col].mode()[0] if len(df[col].mode()) > 0 else 'UNKNOWN'        df[col].fillna(mode_val, inplace=True)remaining_missing = df.isnull().sum().sum()print(f'\nRemaining missing values after imputation: {remaining_missing}')

## 5. Feature EngineeringDerived structural, traffic, and interaction features as outlined in our proposal.

In [ ]:
# Core engineered featuresdf['BRIDGE_AGE'] = df['YEAR'] - df['YEAR_BUILT_027']df['AGE_TO_SPAN_RATIO'] = df['BRIDGE_AGE'] / df['MAX_SPAN_LEN_MT_048'].replace(0, np.nan)df['TRAFFIC_DENSITY'] = df['ADT_029'] / df['STRUCTURE_LEN_MT_049'].replace(0, np.nan)df['DECK_TO_ROADWAY_RATIO'] = df['DECK_WIDTH_MT_052'] / df['ROADWAY_WIDTH_MT_051'].replace(0, np.nan)df['ADT_GROWTH_RATIO'] = df['FUTURE_ADT_114'] / df['ADT_029'].replace(0, np.nan)# Reconstruction featuresdf['WAS_RECONSTRUCTED'] = (df['YEAR_RECONSTRUCTED_106'] > 0).astype(int)df['TIME_SINCE_RECONSTRUCTION'] = np.where(    df['YEAR_RECONSTRUCTED_106'] > 0,    df['YEAR'] - df['YEAR_RECONSTRUCTED_106'],    df['BRIDGE_AGE']  # If never reconstructed, use full age)# Interaction termsif 'SCOUR_CRITICAL_113' in df.columns:    df['AGE_X_SCOUR'] = df['BRIDGE_AGE'] * df['SCOUR_CRITICAL_113']# Handle infinities from divisiondf.replace([np.inf, -np.inf], np.nan, inplace=True)eng_cols = ['BRIDGE_AGE', 'AGE_TO_SPAN_RATIO', 'TRAFFIC_DENSITY', 'DECK_TO_ROADWAY_RATIO',            'ADT_GROWTH_RATIO', 'WAS_RECONSTRUCTED', 'TIME_SINCE_RECONSTRUCTION', 'AGE_X_SCOUR']for col in eng_cols:    if col in df.columns and df[col].isnull().any():        df[col].fillna(df[col].median(), inplace=True)print('Engineered features created:')for col in eng_cols:    if col in df.columns:        print(f'  ✅ {col:35s} mean={df[col].mean():.2f}  std={df[col].std():.2f}')print(f'\nFinal shape after engineering: {df.shape}')

## 6. Exploratory Data Analysis### 6a. Bridge Age Distribution by Condition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))colors = {'Critical': '#e74c3c', 'Poor': '#e67e22', 'Fair': '#f39c12', 'Good': '#27ae60'}order = ['Critical', 'Poor', 'Fair', 'Good']# Histogramfor cond in order:    subset = df[df['TARGET_CONDITION'] == cond]['BRIDGE_AGE']    axes[0].hist(subset, bins=50, alpha=0.6, label=cond, color=colors[cond], edgecolor='black')axes[0].set_title('Bridge Age Distribution by Condition', fontsize=14, fontweight='bold')axes[0].set_xlabel('Bridge Age (years)')axes[0].set_ylabel('Count')axes[0].legend()axes[0].set_xlim(0, 150)# Box plotdata_for_box = [df[df['TARGET_CONDITION'] == c]['BRIDGE_AGE'] for c in order]bp = axes[1].boxplot(data_for_box, labels=order, patch_artist=True)for patch, color in zip(bp['boxes'], [colors[c] for c in order]):    patch.set_facecolor(color)axes[1].set_title('Bridge Age by Condition Class', fontsize=14, fontweight='bold')axes[1].set_ylabel('Bridge Age (years)')plt.tight_layout()plt.savefig('plots/03_bridge_age.png', dpi=150, bbox_inches='tight')plt.show()print('INSIGHT: Older bridges are significantly more likely to be in Critical/Poor condition.')

### 6b. Average Daily Traffic (ADT) by Condition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))# Log-transformed ADT box plotdata_for_box = [np.log1p(df[df['TARGET_CONDITION'] == c]['ADT_029']) for c in order]bp = axes[0].boxplot(data_for_box, labels=order, patch_artist=True)for patch, color in zip(bp['boxes'], [colors[c] for c in order]):    patch.set_facecolor(color)axes[0].set_title('Log(ADT) by Condition Class', fontsize=14, fontweight='bold')axes[0].set_ylabel('Log(Average Daily Traffic)')# Traffic densitydata_for_box = [df[df['TARGET_CONDITION'] == c]['TRAFFIC_DENSITY'].clip(0, 500) for c in order]bp = axes[1].boxplot(data_for_box, labels=order, patch_artist=True)for patch, color in zip(bp['boxes'], [colors[c] for c in order]):    patch.set_facecolor(color)axes[1].set_title('Traffic Density (ADT / Bridge Length) by Condition', fontsize=14, fontweight='bold')axes[1].set_ylabel('Traffic Density')plt.tight_layout()plt.savefig('plots/04_traffic_analysis.png', dpi=150, bbox_inches='tight')plt.show()print('INSIGHT: Traffic volume alone does not strongly separate conditions — heavier traffic bridges get more maintenance funding.')

### 6c. Structure Material Type vs Condition

In [ ]:
# STRUCTURE_KIND_043A: 1=Concrete, 2=Concrete continuous, 3=Steel, 4=Steel continuous,#                       5=Prestressed concrete, 6=Prestressed concrete continuous, 7=Wood, 8=Masonry, 9=Aluminum/Ironkind_labels = {0: 'Other', 1: 'Concrete', 2: 'Concrete Cont.', 3: 'Steel',               4: 'Steel Cont.', 5: 'Prestressed', 6: 'Prestressed Cont.',               7: 'Wood', 8: 'Masonry', 9: 'Aluminum/Iron'}if 'STRUCTURE_KIND_043A' in df.columns:    df_kind = df.copy()    df_kind['Material'] = df_kind['STRUCTURE_KIND_043A'].map(kind_labels).fillna('Other')        ct = pd.crosstab(df_kind['Material'], df_kind['TARGET_CONDITION'], normalize='index')[order] * 100    ct = ct.sort_values('Critical', ascending=False)        fig, ax = plt.subplots(figsize=(14, 7))    ct.plot(kind='barh', stacked=True, color=[colors[c] for c in order], ax=ax, edgecolor='black')    ax.set_title('Condition Distribution by Bridge Material Type', fontsize=14, fontweight='bold')    ax.set_xlabel('Percentage (%)')    ax.legend(title='Condition', bbox_to_anchor=(1.02, 1))    plt.tight_layout()    plt.savefig('plots/05_material_type.png', dpi=150, bbox_inches='tight')    plt.show()    print('INSIGHT: Wood and masonry bridges have the highest rates of Critical/Poor condition.')

### 6d. Scour Critical Rating vs Condition

In [ ]:
if 'SCOUR_CRITICAL_113' in df.columns:    fig, ax = plt.subplots(figsize=(14, 7))    ct = pd.crosstab(df['SCOUR_CRITICAL_113'], df['TARGET_CONDITION'], normalize='index')[order] * 100    ct.plot(kind='bar', stacked=True, color=[colors[c] for c in order], ax=ax, edgecolor='black')    ax.set_title('Condition Distribution by Scour Critical Rating', fontsize=14, fontweight='bold')    ax.set_xlabel('Scour Critical Rating (lower = more vulnerable)')    ax.set_ylabel('Percentage (%)')    ax.legend(title='Condition', bbox_to_anchor=(1.02, 1))    plt.tight_layout()    plt.savefig('plots/06_scour_rating.png', dpi=150, bbox_inches='tight')    plt.show()    print('INSIGHT: Low scour ratings (0-3) correlate strongly with Critical/Poor bridge condition.')

### 6e. State-Level Analysis

In [ ]:
# Calculate % of Critical bridges by statestate_crit = df.groupby('STATE_CODE_001')['TARGET_CONDITION'].apply(    lambda x: (x == 'Critical').mean() * 100).sort_values(ascending=False).head(20)fig, ax = plt.subplots(figsize=(14, 7))state_crit.plot(kind='barh', color='coral', edgecolor='black', ax=ax)ax.set_title('Top 20 States by % of Critical Bridges', fontsize=14, fontweight='bold')ax.set_xlabel('% Critical Bridges')ax.set_ylabel('State Code')plt.tight_layout()plt.savefig('plots/07_state_critical.png', dpi=150, bbox_inches='tight')plt.show()print('INSIGHT: States vary wildly in bridge condition — reflects maintenance budgets and climate.')

### 6f. Year-over-Year Condition Trends

In [ ]:
yearly = pd.crosstab(df["YEAR"], df["TARGET_CONDITION"], normalize="index")[order] * 100

fig, ax = plt.subplots(figsize=(14, 7))

# Robust plotting logic: handle single-year data by defaulting to bar chart
if len(yearly) > 1:
    yearly.plot(kind="area", stacked=True, color=[colors[c] for c in order], alpha=0.8, ax=ax)
    ax.set_title("Bridge Condition Trends Over Time", fontsize=14, fontweight="bold")
else:
    yearly.plot(kind="bar", stacked=True, color=[colors[c] for c in order], edgecolor="black", ax=ax)
    ax.set_title("Bridge Condition Distribution (Single Year Captured: " + str(yearly.index[0]) + ")", fontsize=14, fontweight="bold")

ax.set_xlabel("Year")
ax.set_ylabel("Percentage (%)")
ax.legend(title="Condition", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.savefig("plots/08_yearly_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("INSIGHT: For a mult-year trend, increase SAMPLE_SIZE in the loading cell to >1,000,000 records.")

### 6g. Correlation Heatmap (Top Features)

In [ ]:
# Encode target as ordinal for correlationtarget_map = {'Critical': 0, 'Poor': 1, 'Fair': 2, 'Good': 3}df['TARGET_NUM'] = df['TARGET_CONDITION'].map(target_map)# Get top correlated numeric featuresnum_df = df.select_dtypes(include='number')corr_with_target = num_df.corr()['TARGET_NUM'].drop('TARGET_NUM').abs().sort_values(ascending=False)top_features = corr_with_target.head(20).index.tolist()print('Top 20 features correlated with target:')for i, feat in enumerate(top_features):    print(f'  {i+1:2d}. {feat:45s} |r| = {corr_with_target[feat]:.4f}')# Heatmap of inter-correlations among top featuresfig, ax = plt.subplots(figsize=(16, 14))corr_matrix = df[top_features + ['TARGET_NUM']].corr()mask = np.triu(np.ones_like(corr_matrix, dtype=bool))sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',            center=0, square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)ax.set_title('Correlation Heatmap — Top 20 Features vs Target', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('plots/09_correlation_heatmap.png', dpi=150, bbox_inches='tight')plt.show()

### 6h. Multicollinearity Check (|r| > 0.90)

In [ ]:
# Find highly correlated feature pairscorr_matrix_full = df[top_features].corr().abs()upper = corr_matrix_full.where(np.triu(np.ones(corr_matrix_full.shape), k=1).astype(bool))high_corr_pairs = []for col in upper.columns:    for idx in upper.index:        if upper.loc[idx, col] > 0.90:            high_corr_pairs.append((idx, col, upper.loc[idx, col]))if high_corr_pairs:    print('⚠️  Highly correlated feature pairs (|r| > 0.90):')    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: -x[2]):        print(f'  {f1} ↔ {f2}: r = {r:.4f}  → Consider dropping one')else:    print('✅ No feature pairs with |r| > 0.90 found among top features.')# Drop one from each highly correlated paircols_to_remove = set()for f1, f2, r in high_corr_pairs:    # Keep the one more correlated with target    if corr_with_target.get(f1, 0) >= corr_with_target.get(f2, 0):        cols_to_remove.add(f2)    else:        cols_to_remove.add(f1)if cols_to_remove:print(f'\nDropping {len(cols_to_remove)} redundant features: {cols_to_remove}')    df.drop(columns=[c for c in cols_to_remove if c in df.columns], inplace=True)    print(f'Shape after multicollinearity filtering: {df.shape}')

### 6i. Engineered Features Analysis

In [ ]:
eng_features = ['BRIDGE_AGE', 'AGE_TO_SPAN_RATIO', 'TRAFFIC_DENSITY',                'TIME_SINCE_RECONSTRUCTION', 'WAS_RECONSTRUCTED', 'AGE_X_SCOUR']eng_features = [f for f in eng_features if f in df.columns]fig, axes = plt.subplots(2, 3, figsize=(20, 12))axes = axes.flatten()for i, feat in enumerate(eng_features):    if i < len(axes):        data = [df[df['TARGET_CONDITION'] == c][feat].clip(            df[feat].quantile(0.01), df[feat].quantile(0.99)        ) for c in order]        bp = axes[i].boxplot(data, labels=order, patch_artist=True)        for patch, color in zip(bp['boxes'], [colors[c] for c in order]):            patch.set_facecolor(color)        axes[i].set_title(feat, fontsize=12, fontweight='bold')        axes[i].tick_params(axis='x', rotation=15)# Hide unused subplotsfor j in range(len(eng_features), len(axes)):    axes[j].set_visible(False)plt.suptitle('Engineered Features by Condition Class', fontsize=16, fontweight='bold', y=1.01)plt.tight_layout()plt.savefig('plots/10_engineered_features.png', dpi=150, bbox_inches='tight')plt.show()print('INSIGHT: BRIDGE_AGE and AGE_X_SCOUR show the clearest separation between condition classes.')

## 7. Final Dataset Summary

In [ ]:
print('=' * 60)print('FINAL CLEANED DATASET SUMMARY')print('=' * 60)print(f'Total rows:        {len(df):,}')print(f'Total columns:     {df.shape[1]}')print(f'Numeric features:  {df.select_dtypes(include="number").shape[1]}')print(f'Categorical cols:  {df.select_dtypes(include="object").shape[1]}')print(f'Missing values:    {df.isnull().sum().sum()}')print(f'Memory:            {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')print(f'\nTarget Distribution:')for cond in order:    count = (df['TARGET_CONDITION'] == cond).sum()    pct = count / len(df) * 100    print(f'  {cond:10s}: {count:>10,}  ({pct:.2f}%)')print('=' * 60)

## 8. Save Cleaned Dataset

In [ ]:
# Save the cleaned dataset for modelingoutput_file = 'nbi_cleaned.csv'df.to_csv(output_file, index=False)print(f'Cleaned dataset saved to: {output_file}')print(f'File size: {pd.io.common.file_exists(output_file)}')# Also save as parquet for faster loading during modelingtry:    df.to_parquet('nbi_cleaned.parquet', index=False)    print('Also saved as nbi_cleaned.parquet (faster for model training)')except ImportError:    print('Install pyarrow for parquet: pip install pyarrow')print('✅ EDA and Data Cleaning complete! Ready for model training.')